# Сбор данных с Web-scraping и API для гуманитарных исследований
## Блок 1. API
## Занятие 2. Все данные по запросу: пагинация, ошибки, ключи

На прошлом занятии запрос к API возвращал только первую порцию результатов, а в ответе оставался ключ `continue`: это означало, что данных больше, чем пришло. На этом занятии сбор доводится до конца. Сначала на знакомом материале Википедии разбирается, как получить все порции и что делать, если запрос не удался. Затем те же приёмы переносятся на новый API, у которого свои параметры и свои правила доступа, и собранный набор данных проверяется так, как его проверял бы исследователь.

**Материал:** MediaWiki API (русская Википедия) и Open Library — открытый библиотечный каталог, который ведёт некоммерческая организация Internet Archive.

**Исследовательская задача.** Как творчество А.П. Чехова представлено в открытом библиотечном каталоге: какие книги его и о нём там описаны, на каких языках и когда они были впервые изданы? К концу занятия у вас будет таблица со всеми записями Open Library по этому запросу. Однако таблица ещё не даёт ответа на вопрос. Число записей зависит в первую очередь от того, как записано его имя, откуда каталог получает описания и насколько эти описания полны. Путь от вопроса до проверенного набора данных вам предстоит пройти самостоятельно в домашнем задании 1.

**Путь занятия:**

1. повторение: запрос и ответ, словари и списки, циклы;
2. сбор всех порций данных из Википедии;
3. обработка запроса, который не удался;
4. новый API: где найти параметры, какие у него правила доступа, зачем нужны ключи;
5. сбор всех записей из Open Library;
6. превращение записей в таблицу;
7. проверка набора данных: что в таблице говорит о Чехове, а что — о каталоге;
8. ⭐ практикум продвинутого уровня.

---

Ячейки запускаются сверху вниз, потому что каждая часть опирается на предыдущую. Задания для самостоятельной работы отмечены значком ✏️; код пишется в ячейке под заданием на месте комментария `# ВАШ КОД`. Остальные значки означают следующее:

- 🔁 — блоки повторения. Они освежают то, что изучалось на занятии 1 и в курсе «Цифровые технологии в профессиональной деятельности».
- 📨 — задания предполагают ваш выбор и рассуждение. Их результат отправляется в телеграм-чат курса, чтобы его можно было обсудить вместе.
- ⭐ — части и задания продвинутого уровня. На базовом уровне их можно пропустить: остальная тетрадка от них не зависит.

У каждого задания есть значок, который показывает, можно ли пользоваться ИИ-ассистентом:

- 🚫 — без ИИ;
- 💬 — ассистента можно спрашивать только об ошибках и непонятных местах, а код вы пишете сами;
- 🤖 — ИИ разрешён; в журнал `AI_LOG.md` записывается, что вы спрашивали, что приняли из ответа и что исправили.

В первом блоке курса (занятия 1–2 и домашнее задание 1) код не генерируется с помощью ИИ, поэтому задания отмечены 🚫 или 💬. Сначала нужно научиться самостоятельно читать ответ сервера и собственный код: без этого невозможно проверить и код, предложенный ассистентом. Задания с 🤖 появятся начиная с занятия 3. Все задания курса можно выполнить без ИИ.

---
## Часть 0. Подготовка

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

In [ ]:
WIKI_API = "https://ru.wikipedia.org/w/api.php"

# Без осмысленного User-Agent Википедия отвечает кодом 403 (занятие 1)
HEADERS = {
    "User-Agent": "miniDH-scraping-course/1.0 (https://github.com/Minidegree-DH/miniDH-scraping)"
}

---
## Часть 1. 🔁 Повторение

### Запрос и ответ

На занятии 1 запрос к API собирался из трёх частей: адреса, словаря параметров и заголовков. Ответ превращался в словарь методом `.json()`. Ниже повторяется запрос, который возвращает статьи из категории Википедии.

In [ ]:
params = {
    "action": "query",
    "list": "categorymembers",
    "cmtitle": "Категория:Музеи Москвы",
    "cmtype": "page",
    "format": "json",
}

response = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=10)

response.status_code

In [ ]:
# Ответ в формате JSON превращается в словарь Python
data = response.json()

data.keys()

### Откуда берутся параметры

Словарь `params` легко принять за готовую формулу, которую достаточно переписать. На самом деле каждый параметр выбран по документации, и у большинства из них есть другие допустимые значения. Ниже разбирается, что означает каждый параметр этого запроса.

| Параметр | Значение в запросе | Что задаёт | Другие варианты |
|---|---|---|---|
| `action` | `query` | какое действие выполнить: `query` получает данные | `parse` (разобрать текст статьи), `opensearch` (подсказки поиска) и другие |
| `list` | `categorymembers` | какой список нужен | `search` (поиск по тексту, занятие 1), `allpages` (все страницы) и другие |
| `cmtitle` | `Категория:Музеи Москвы` | из какой категории брать страницы | `cmpageid` — номер страницы категории вместо названия |
| `cmtype` | `page` | какие элементы категории вернуть | `subcat` (подкатегории), `file` (файлы); по умолчанию возвращаются все три вида |
| `format` | `json` | в каком формате прислать ответ | `xml`; кроме того, параметр `formatversion=2` делает JSON компактнее |

Почему названия параметров начинаются с `cm`? Список `categorymembers` относится к одному из модулей API, и у каждого модуля есть собственная приставка: у `categorymembers` она `cm`, у `search` — `sr`. Приставка нужна, чтобы параметры разных модулей не путались, если в одном запросе соединить несколько модулей.

Узнать параметры можно тремя способами. Первый способ — [страница справки модуля](https://ru.wikipedia.org/w/api.php?action=help&modules=query%2Bcategorymembers), которая открывается в браузере. Второй — [песочница API](https://ru.wikipedia.org/wiki/Special:ApiSandbox): в ней запрос собирается из выпадающих списков, а результат сразу виден. Третий способ показывает, что MediaWiki API может описать собственные параметры: для этого предназначено действие `paraminfo`.

In [ ]:
# API описывает собственный модуль: действие paraminfo
info_params = {
    "action": "paraminfo",
    "modules": "query+categorymembers",
    "format": "json",
}

info = requests.get(WIKI_API, params=info_params, headers=HEADERS, timeout=10).json()

In [ ]:
info.keys()

In [ ]:
info['paraminfo'].keys()

In [ ]:
# Описание модуля лежит в первом элементе списка modules
module = info["paraminfo"]["modules"][0]

module.keys()

In [ ]:
# Приставка модуля: её нужно добавить к имени каждого параметра
module["prefix"]

In [ ]:
# Каждый параметр описан словарём: имя, тип или список допустимых значений, значение по умолчанию.
# Если у параметра нет значения по умолчанию, в таблице будет NaN
pd.DataFrame(module["parameters"])[["name", "type", "default"]]

Из таблицы можно узнать, например, что у `limit` (то есть у параметра `cmlimit`) значение по умолчанию равно 10, а у `type` по умолчанию указаны все три вида элементов. Параметр `sort` позволяет упорядочить статьи не по алфавиту, а по времени добавления в категорию; параметр `prop` со значением `timestamp` добавляет к каждой статье это время.

#### ✏️🚫 Задание 1

1. Выведите, сколько статей пришло в ответе на первый запрос, и перечислите их названия. Где в ответе лежит список статей? Есть ли в ответе ключ `continue`?

In [ ]:
# ВАШ КОД

2. Пользуясь таблицей параметров, измените запрос так, чтобы получить подкатегории «Категории:Музеи Москвы», а не статьи. Выведите их названия.

In [ ]:
# ВАШ КОД

В первом ответе пришло 10 статей, потому что значение `cmlimit` по умолчанию равно 10; наибольшее допустимое значение равно 500, а для зарегистрированных ботов — 5000. Ключ `continue` означает, что статей в категории больше. Чтобы получить остальные, понадобятся несколько приёмов Python, которые стоит освежить.

### Списки: `append` и `extend`

Метод `.append()` добавляет в список один элемент, а метод `.extend()` добавляет по очереди все элементы другого списка.

In [ ]:
museums = ["Третьяковская галерея"]

# append: один новый элемент в конец списка
museums.append("Музей Москвы")
museums

In [ ]:
# extend: все элементы другого списка по очереди
museums.extend(["Дарвиновский музей", "Музей Маяковского"])
museums

In [ ]:
# Если передать список в append, он добавится целиком, как один элемент
museums.append(["Музей А", "Музей Б"])
museums

In [ ]:
# Последний элемент — список внутри списка
museums[-1]

### Словари: `get` и `update`

Метод `.get()` возвращает значение по ключу. Если ключа нет, метод возвращает `None` или указанное значение по умолчанию, а не вызывает ошибку. Метод `.update()` добавляет в словарь пары из другого словаря; если ключ уже был в словаре, его значение заменяется.

In [ ]:
book = {"title": "Вишнёвый сад", "year": 1904}

# Обращение в квадратных скобках и get дают одно и то же, если ключ есть
print(book["year"])
print(book.get("year"))

In [ ]:
# Если ключа нет, get возвращает None, а квадратные скобки вызвали бы KeyError
print(book.get("author"))

# Второй аргумент get — значение на случай, если ключа нет
print(book.get("author", "неизвестен"))

In [ ]:
# update: ключа author не было — он добавится; ключ year был — значение заменится
book.update({"author": "А. П. Чехов", "year": 1903})
book

### Циклы: `for`, `while`, `break`

Цикл `for` перебирает элементы последовательности, например `range(5)`, то есть числа от 0 до 4. Цикл `while` повторяется, пока условие истинно. Команда `break` прерывает цикл досрочно.

In [ ]:
# range(5) — числа от 0 до 4
for number in range(5):
    print(number)

In [ ]:
for number in range(5):
    if number == 3:
        break  # цикл прерывается, не дойдя до конца
    print(number)

In [ ]:
total = 0

# while повторяется, пока условие истинно
while total < 10:
    total += 4  # то же, что total = total + 4
    print(total)

#### ✏️🚫 Задание 2

Ниже дан список «порций», как будто они приходили от сервера по очереди. Пройдите по нему циклом `for` и соберите все элементы в один список `collected` с помощью метода `extend`. Остановите цикл командой `break`, как только встретится пустая порция. Выведите `collected`.

In [ ]:
portions = [["а", "б"], ["в", "г"], ["д"], [], ["е"]]

collected = []

# ВАШ КОД

---
## Часть 2. Все порции данных из Википедии

### Зачем API отдаёт данные порциями

Сервер не присылает тысячи записей одним ответом, потому что такой ответ долго готовить и долго передавать. Поэтому почти любой API отдаёт результаты порциями (их называют также страницами), и, чтобы собрать всё, порции запрашивают одну за другой. Этот приём называется **пагинацией**.

В MediaWiki следующую порцию указывает сам сервер: в блоке `continue` он присылает метку, с которой нужно продолжить. Правило сформулировано в [документации](https://www.mediawiki.org/wiki/API:Continue): всё содержимое `continue` добавляется к параметрам следующего запроса без изменений. Когда ключа `continue` в ответе нет, данные кончились.

### Два шага вручную

Для наглядности ниже запрашиваются порции по 50 статей. В «Категории:Музеи Москвы» около сотни статей, поэтому понадобится несколько запросов.

In [ ]:
params = {
    "action": "query",
    "list": "categorymembers",
    "cmtitle": "Категория:Музеи Москвы",
    "cmtype": "page",
    "cmlimit": 50,
    "format": "json",
}

response = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=10)
first = response.json()

first.keys()

In [ ]:
# Сколько статей пришло в первой порции
len(first["query"]["categorymembers"])

In [ ]:
# Что лежит в блоке continue
first["continue"]

В блоке `continue` два ключа. Ключ `cmcontinue` содержит метку, с которой нужно продолжить; разбирать её не требуется, достаточно вернуть серверу. Ключ `continue` содержит служебное значение для самого API. Оба ключа добавляются к параметрам методом `.update()`.

In [ ]:
# Содержимое continue добавляется к параметрам
params.update(first["continue"])

params

In [ ]:
# Второй запрос отправляется с обновлёнными параметрами
response = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=10)
second = response.json()

len(second["query"]["categorymembers"])

In [ ]:
# Последняя статья первой порции и первая статья второй: список продолжается, а не начинается заново
print(first["query"]["categorymembers"][-1]["title"])
print(second["query"]["categorymembers"][0]["title"])

In [ ]:
# Есть ли ещё порции
"continue" in second

### Цикл

Повторять эти шаги вручную для категории из тысяч статей невозможно, поэтому те же действия записываются циклом. На каждом проходе цикла выполняются одни и те же действия:

1. отправить запрос;
2. добавить пришедшие статьи в общий список (`extend`);
3. если ключа `continue` нет, выйти из цикла (`break`);
4. в противном случае добавить содержимое `continue` к параметрам (`update`) и сделать паузу.

Число проходов цикла нужно ограничить. Если в коде окажется ошибка и ключ `continue` будет приходить всегда, цикл без ограничителя никогда не остановится и будет нагружать чужой сервер. Поэтому вместо `while True` удобнее записать `for _ in range(max_requests)`: такой цикл выполнится не больше `max_requests` раз. Подчёркивание вместо имени переменной означает, что номер прохода в цикле не используется.

Размер порции тоже выбирается сознательно. Чем больше `cmlimit`, тем меньше запросов придётся отправить, но тем больше становится каждый ответ. Для категорий Википедии разумно запрашивать от 50 до 500 статей за раз.

#### ✏️💬 Задание 3

Допишите функцию `get_all_category_members()` по четырём шагам, перечисленным выше. Проверьте её на «Категории:Музеи Москвы»: число статей должно совпасть с числом, указанным на странице категории в браузере. Затем соберите статьи категории из вашей области.

In [ ]:
def get_all_category_members(category: str, limit: int = 50, max_requests: int = 20) -> list[dict]:
    """Возвращает все статьи категории Википедии."""
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": category,
        "cmtype": "page",
        "cmlimit": limit,
        "format": "json",
    }
    members = []

    for _ in range(max_requests):
        # 1. отправить запрос
        # 2. добавить статьи в members
        # 3. если continue нет — выйти из цикла
        # 4. иначе обновить параметры и сделать паузу
        ...

    return members


museums = get_all_category_members("Категория:Музеи Москвы")
len(museums)

В коде функции нигде не разбирается, как устроена метка `cmcontinue`: серверу возвращается то, что он прислал. Поэтому тот же цикл подходит и для других запросов к MediaWiki.

Этот способ пагинации называется **курсорным**, потому что следующую порцию указывает сервер. В других API встречаются ещё два способа: номер страницы (`page=2`) и смещение (`offset=100`, то есть «пропустить первые сто записей»). Смещение понадобится в части 5.

#### ✏️🔁📨💬 Задание 4

Составьте список из двух-трёх категорий Википедии, относящихся к вашей области. Для каждой категории соберите все статьи функцией `get_all_category_members()`, превратите результат в таблицу и сохраните её в отдельный CSV-файл. Имя файла составьте из названия категории без слова «Категория:». Выведите название каждой категории и число статей в ней.

📨 Отправьте в телеграм-чат курса свой код и число статей в каждой категории.

In [ ]:
categories = []

# ВАШ КОД

---
## Часть 3. Если запрос не удался

### 🔁 Статус-коды

На занятии 1 разбирались статус-коды ответа: код `200` означает успех, коды, начинающиеся с 4, означают ошибку клиента, а коды, начинающиеся с 5, — ошибку сервера. Метод `.raise_for_status()` прерывает выполнение, если код начинается с 4 или 5, чтобы программа не продолжила работу с ошибочным ответом.

Бывает, однако, что ответа нет вовсе: сервер не ответил за отведённое время или соединение не установилось. Тогда `requests.get()` не возвращает ответ, а вызывает **исключение**, то есть ошибку, которая останавливает программу.

Отведённое время задаёт аргумент `timeout`. Запись `timeout=10` означает, что программа ждёт соединения и очередной части ответа не дольше 10 секунд. Если `timeout` не указать, программа может ждать ответа сколь угодно долго. Можно задать и два числа, например `timeout=(3, 30)`: первое ограничивает ожидание соединения, второе — ожидание данных. Так поступают, когда сервер отвечает быстро, но присылает большие ответы.

In [ ]:
# Адрес несуществующего сервера: запрос завершится исключением.
# Раскомментируйте строку, выполните ячейку и посмотрите на сообщение об ошибке,
# затем закомментируйте снова, чтобы тетрадка выполнялась сверху вниз без остановок.
# requests.get("https://ru.wikipedia.invalid/w/api.php", timeout=10)

### 🔁 `try` и `except`

Исключение можно перехватить. Код, в котором может возникнуть ошибка, помещают в блок `try`. Если ошибка указанного типа возникла, выполнение переходит в блок `except`, и программа не останавливается.

In [ ]:
try:
    # Если здесь возникнет ошибка, следующая строка уже не выполнится
    requests.get("https://ru.wikipedia.invalid/w/api.php", timeout=10)
    print("Ответ получен")
except requests.exceptions.ConnectionError:
    # Сюда выполнение попадает, только если возникла ошибка соединения
    print("Не удалось соединиться с сервером")

print("Программа продолжает работу")

### Повторная попытка

Обрыв связи или перегрузка сервера обычно длятся недолго, поэтому неудачный запрос имеет смысл повторить через несколько секунд. Без повторных попыток длинный сбор может оборваться на середине, и тогда его придётся начинать сначала.

Повторять стоит не любую ошибку. Если сервера нет, связь оборвалась или сервер просит подождать (код `429` означает «слишком много запросов», код `503` — «сервер перегружен»), повтор может помочь. Если запрос составлен неверно (`400`), доступа нет (`403`) или страницы нет (`404`), повторный запрос получит тот же ответ: в этом случае нужно исправлять запрос, а не повторять его.

### Лимиты Wikimedia

Код `429` встречается на практике. На занятии 1 выяснилось, что без заголовка User-Agent Википедия отвечает кодом `403`. С 2026 года Wikimedia Foundation, кроме того, вводит общие лимиты на число запросов ко всем своим API. На странице [Wikimedia APIs/Rate limits](https://www.mediawiki.org/wiki/Wikimedia_APIs/Rate_limits) указано следующее:

- запросам, о которых известен только IP-адрес, разрешено 500 запросов в час;
- запросы с осмысленным заголовком User-Agent получают более высокий лимит, но точного числа на странице нет;
- одновременно рекомендуется отправлять не больше трёх запросов;
- при превышении лимита сервер отвечает кодом `429`; в заголовке ответа `Retry-After` обычно указано, сколько секунд нужно подождать, а если такого заголовка нет, следует подождать не меньше 5 секунд.

На той же странице сказано, что лимиты ещё уточняются, поэтому цифры стоит перепроверять; здесь они приведены по состоянию на ноябрь 2026 года. Из этих правил для занятия следуют два вывода. Во-первых, в аудитории все ноутбуки обычно выходят в интернет через один IP-адрес, и лимит, рассчитанный на одного пользователя, делится на всех. Поэтому паузы между запросами и заголовок `HEADERS` с адресом репозитория курса нужны и здесь. Во-вторых, на код `429` правильно ответить паузой и повтором, а не немедленным новым запросом.

### Функция с повторными попытками

Функция ниже делает до трёх попыток с паузой в 5 секунд и выводит сообщение о каждой неудаче. Число попыток и длина паузы выбраны по рекомендации Wikimedia; у других API рекомендации бывают иными, и их ищут в документации. Распространён и другой подход, при котором пауза после каждой неудачи удваивается (5, 10, 20 секунд и так далее): так программа меньше нагружает сервер, которому нужно время, чтобы восстановиться.

Аргумент `headers` по умолчанию равен `HEADERS`. Другой заголовок понадобится в части 4, где у API свои требования к User-Agent.

In [ ]:
def get_json(url: str, params: dict, attempts: int = 3, headers: dict = HEADERS) -> dict:
    """Отправляет GET-запрос и возвращает JSON; при временных сбоях повторяет запрос."""
    # range(1, attempts + 1) — номера попыток начиная с единицы: 1, 2, 3
    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
        # В скобках — два типа ошибок: перехватывается любая из них
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            print(f"Попытка {attempt}: нет ответа от сервера")
            time.sleep(5)
            continue  # к следующей попытке

        # Ответ пришёл, но сервер просит подождать: запрос тоже повторяется
        if response.status_code in (429, 503):
            print(f"Попытка {attempt}: код {response.status_code}, сервер просит подождать")
            time.sleep(5)
            continue

        response.raise_for_status()  # 400, 403, 404: повтор не поможет
        return response.json()       # успех: return сразу завершает функцию

    # Сюда выполнение доходит, только если все попытки закончились неудачей
    raise RuntimeError(f"Не удалось получить ответ за {attempts} попытки: {url}")

Команда `continue` внутри цикла (её не следует путать с ключом `continue` в ответе MediaWiki) пропускает остаток прохода и переходит к следующему. Команда `raise` в конце функции вызывает исключение, если все попытки исчерпаны. Лучше остановить программу с понятным сообщением, чем молча вернуть пустой результат, который потом будет принят за настоящий.

#### ✏️💬 Задание 5

1. Вызовите `get_json()` с адресом `https://ru.wikipedia.invalid/w/api.php` и любыми параметрами. Сколько попыток делает функция и чем заканчивается вызов?
2. Замените в функции `get_all_category_members()` две строки, а именно запрос и превращение ответа в словарь, одной строкой: `data = get_json(WIKI_API, params)`. Убедитесь, что результат не изменился.

In [ ]:
# ВАШ КОД

---
## Часть 4. Новый API: Open Library

### Где искать данные о книгах

Википедия — энциклопедия, и из неё можно узнать, о ком и о чём написаны статьи. Исследовательский вопрос занятия, однако, другой: какие книги Чехова и книги о нём описаны в библиотечных каталогах. Для ответа нужны библиографические данные, то есть сведения об авторе, названии, языке, издателе и годе издания.

Такие данные есть в [Open Library](https://openlibrary.org/) — открытом каталоге, который ведёт некоммерческая организация Internet Archive. Цель проекта состоит в том, чтобы завести страницу для каждой изданной книги. Описания поступают из каталогов библиотек, от книготорговцев и из оцифровки, которую ведёт сам Internet Archive, а дополнять и исправлять их, как статьи Википедии, может любой зарегистрированный пользователь.

Для исследователя это важно с самого начала: в Open Library описано то, что было передано в каталог или внесено в него, и так, как это было записано. К этому придётся вернуться в части 7.

Open Library различает **произведение** (work) и **издание** (edition). «Вишнёвый сад» — одно произведение, а его переводы и переиздания — разные издания этого произведения. Поиск возвращает произведения, а сколько у произведения изданий, указано в отдельном поле.

### ⭐ Произведение и издание: модель FRBR

Различение произведения и издания пришло в Open Library из библиотечной теории. В 1998 году Международная федерация библиотечных ассоциаций и учреждений (IFLA) опубликовала модель FRBR (Functional Requirements for Bibliographic Records), в которой описание книги разделено на четыре уровня:

- **произведение** — замысел и содержание: «Вишнёвый сад» Чехова как таковой;
- **выражение** — конкретный текст произведения: русский оригинал, английский перевод Констанс Гарнетт, немецкий перевод;
- **воплощение** — издание выражения: тираж определённого издательства определённого года;
- **экземпляр** — конкретная книга на полке конкретной библиотеки.

В 2017 году IFLA объединила FRBR и родственные модели в модель IFLA LRM (Library Reference Model), но четыре уровня в ней сохранились.

В Open Library уровней только два: произведение и издание, причём издание примерно соответствует воплощению. Уровня выражения нет, поэтому переводы оказываются изданиями того же произведения, а список языков у произведения объединяет оригинал и все переводы. Для вопроса о переводах Чехова это существенно: по записи произведения нельзя узнать, сколько существует разных переводов, можно узнать только, на каких языках выходили издания.

### Как найти API

Как и на занятии 1, ссылку на API ищут в подвале сайта. Внизу страницы openlibrary.org есть раздел Develop с четырьмя ссылками: Developer Center (общие правила), API Documentation (описание всех API), Bulk Data Dumps (выгрузки всего каталога) и Writing Bots (правила для программ, которые редактируют каталог). Поиск описан на странице [Search API](https://openlibrary.org/dev/docs/api/search), а его эндпоинт выглядит так:

`https://openlibrary.org/search.json`

### Правила доступа

Прежде чем отправлять запросы, нужно прочитать правила. В [Developer Center](https://openlibrary.org/developers/api) сказано следующее (по состоянию на ноябрь 2026 года):

- ключ не нужен;
- без идентификации разрешён 1 запрос в секунду;
- если в заголовке User-Agent указаны название программы и адрес электронной почты или телефон для связи, разрешено 3 запроса в секунду;
- скачивать через API метаданные в больших объёмах не следует, потому что для этого есть ежемесячные выгрузки (Bulk Data Dumps);
- за нарушение правил возможны жёсткое ограничение запросов или блокировка.

Подход здесь тот же, что у Wikimedia (часть 3): владелец API хочет знать, кто нагружает сервер, и иметь возможность связаться с автором программы. Для вопроса занятия понадобится несколько тысяч записей, то есть около десятка запросов, и для этого API подходит. Если бы исследование касалось всего каталога, правильнее было бы скачать выгрузку.

Open Library просит указывать в User-Agent контакт для связи, то есть адрес электронной почты или телефон. Впишите в ячейку ниже адрес, по которому с вами можно связаться, например университетскую почту.

In [ ]:
OPENLIBRARY_API = "https://openlibrary.org/search.json"

# Open Library просит указать в User-Agent название программы и адрес для связи
OL_HEADERS = {
    "User-Agent": "miniDH-scraping-course/1.0 (https://github.com/Minidegree-DH/miniDH-scraping; your-email@example.com)"
}

### Если API требует ключ

Open Library опознаёт пользователя по заголовку User-Agent. Многие другие API выдают пользователю **ключ**, то есть строку, которую нужно передавать с каждым запросом; так устроены, например, API Europeana и Смитсоновского института. По ключу владелец API узнаёт, кто создаёт нагрузку, может ограничить число запросов и связаться с автором скрипта. Обычно ключ бесплатный и выдаётся после регистрации, а способ его передачи (параметр запроса или заголовок) указан в документации.

Если отправить запрос без ключа или с неверным ключом, сервер ответит кодом `401 Unauthorized`, что означает «пользователь не опознан».

Ключ следует хранить так же бережно, как пароль. Если вписать его в тетрадку и отправить её в публичный репозиторий, ключом сможет воспользоваться кто угодно, потому что существуют программы, которые постоянно просматривают GitHub в поисках таких строк. Поэтому ключ хранят в отдельном файле, который не попадает в репозиторий. В файле `.gitignore` репозитория курса есть правило `*.token`, и файлы с таким расширением git не отслеживает.

Ниже определена функция, которая читает ключ из такого файла. Она пригодится в домашнем задании 1, если выбранный API потребует ключ.

In [ ]:
def read_key(filename: str) -> str | None:
    """Читает ключ API из файла; если файла нет, возвращает None."""
    # Path — путь к файлу; сам файл при этом не создаётся и не читается
    key_file = Path(filename)

    if not key_file.exists():
        print(f"Файл {filename} не найден")
        return None

    # read_text читает файл целиком, strip убирает пробелы и перенос строки по краям
    return key_file.read_text(encoding="utf-8").strip()

In [ ]:
# Файла с таким именем рядом с тетрадкой нет, поэтому функция вернёт None
key = read_key("example_api.token")

print(key)

### Токен на практике: песочница httpbin

Чтобы увидеть, как ключ передаётся серверу, не регистрируясь в настоящем сервисе, удобно воспользоваться песочницей. Сервис [httpbin.org](https://httpbin.org/) создан для того, чтобы проверять HTTP-запросы: он отвечает на них, ничего не храня. Эндпоинт `https://httpbin.org/bearer` проверяет, передан ли токен в заголовке `Authorization`. Настоящий сервис выдал бы токен после регистрации и проверял бы, что он действителен; песочница принимает любую строку, поэтому токен можно создать самостоятельно.

Порядок действий здесь тот же, что с настоящим ключом: получить токен, сохранить его в файл с расширением `.token`, прочитать функцией `read_key()` и передать в запросе. Сначала стоит посмотреть, что ответит сервер на запрос без токена.

In [ ]:
SANDBOX_URL = "https://httpbin.org/bearer"

# Запрос без заголовка Authorization: сервер не знает, кто его отправил
response = requests.get(SANDBOX_URL, headers=HEADERS, timeout=10)

response.status_code

Код `401` означает, что пользователь не опознан. Теперь нужно «получить» токен. Песочница токенов не выдаёт, поэтому токен создаётся функцией `secrets.token_hex()` из стандартной библиотеки: она возвращает случайную строку из шестнадцатеричных цифр. Токен записывается в файл `sandbox.token` только один раз; если файл уже есть, используется сохранённый токен.

In [ ]:
import secrets

token_file = Path("sandbox.token")

if not token_file.exists():
    # token_hex(16) — 16 случайных байтов, записанных как 32 шестнадцатеричные цифры
    token_file.write_text(secrets.token_hex(16), encoding="utf-8")
    print("Создан новый токен")
else:
    print("Токен уже сохранён")

In [ ]:
# Токен читается из файла той же функцией, что и любой ключ
token = read_key("sandbox.token")

# Длина токена; сам токен на экран не выводится, как не выводят пароль
len(token)

Токены часто передают в заголовке `Authorization` в виде `Bearer <токен>` («предъявитель токена»). Другие API, как уже говорилось, принимают ключ параметром запроса; какой способ нужен, сказано в документации. Заголовок с токеном добавляется к заголовку `HEADERS` с User-Agent.

In [ ]:
# Новый словарь: User-Agent из HEADERS и заголовок с токеном.
# Запись {**HEADERS, ...} копирует все пары из HEADERS и добавляет новую
auth_headers = {**HEADERS, "Authorization": f"Bearer {token}"}

response = requests.get(SANDBOX_URL, headers=auth_headers, timeout=10)

response.status_code

In [ ]:
# Песочница сообщает, что пользователь опознан, и показывает, какой токен получила
response.json().keys()

In [ ]:
# Совпадает ли полученный сервером токен с сохранённым в файле
response.json()["token"] == token

Файл `sandbox.token` не попадёт в репозиторий благодаря правилу `*.token` в `.gitignore`. Перед коммитом полезно убедиться в этом командой `git status`: файла с токеном в списке изменений быть не должно.

### Параметры поиска

Параметры поиска перечислены на странице Search API. В таблице ниже приведены те из них, которые понадобятся на занятии, и варианты, о которых полезно знать.

| Параметр | Что задаёт | Варианты и замечания |
|---|---|---|
| `q` | поисковый запрос | ищет совпадения во многих полях записи, а не только в имени автора; поддерживает особый синтаксис (см. ниже) |
| `author`, `title` | поиск только по автору или только по названию | удобны, когда известно, в каком поле искать |
| `fields` | какие поля вернуть | по умолчанию приходит много служебных полей; в документации отдельно сказано, что запрос всех полей (`fields=*`) обходится серверу дорого |
| `limit` | сколько записей в порции | предел в документации не указан |
| `offset`, `page` | какую порцию вернуть | два способа пагинации (часть 5) |
| `sort` | порядок записей | по умолчанию записи упорядочены по релевантности; `new` и `old` упорядочивают их по году, `random` — случайным образом, `editions` — по числу изданий |
| `lang` | предпочтительный язык | не отбирает записи, а только поднимает выше издания на указанном языке |

Значение параметра `q` разбирает поисковая система Apache Solr, на которой построен поиск каталога, поэтому в запросе можно указывать поле и условие:

- `language:ger` — только произведения, у которых есть издания на немецком языке;
- `first_publish_year:[1900 TO 1910]` — диапазон лет первого издания;
- `author:Chekhov title:Cherry` — условия на отдельные поля;
- `"Anton Chekhov"` в кавычках — точная фраза;
- `OR` и `NOT` — объединение и исключение; эти слова пишутся заглавными буквами, иначе они ищутся как обычные слова.

Подробное описание синтаксиса есть на странице справки каталога Search HowTo, на которую ссылается документация Search API.

### Первый запрос

Запрос отправляется через уже готовую функцию `get_json()`, но с заголовком `OL_HEADERS`. В параметре `fields` перечислены через запятую только те поля, которые понадобятся для таблицы.

In [ ]:
# Поля, которые понадобятся для таблицы
FIELDS = "key,title,author_name,author_key,first_publish_year,language,publisher,edition_count"

params = {"q": "Chekhov", 
          "limit": 5, 
          "fields": FIELDS}

data = get_json(OPENLIBRARY_API, params, headers=OL_HEADERS)

data.keys()

In [ ]:
# Сколько всего записей найдено
data["numFound"]

In [ ]:
# Записи этой порции — список
print(type(data["docs"]))
print(len(data["docs"]))

Ответ устроен иначе, чем у Википедии: список записей лежит в `docs`, а общее число найденных записей — в `numFound`. Ключа `continue` в ответе нет, потому что у Open Library другой способ пагинации (часть 5).

In [ ]:
# Первая запись — словарь с полями, перечисленными в fields
first_doc = data["docs"][0]

first_doc.keys()

In [ ]:
# Название — строка
first_doc["title"]

In [ ]:
# Автор пришёл списком: у книги может быть несколько авторов
first_doc.get("author_name")

In [ ]:
# Языки — тоже список: это языки всех изданий произведения
first_doc.get("language")

In [ ]:
# Год первого известного каталогу издания — число
first_doc.get("first_publish_year")

Одни поля пришли строками и числами, другие — **списками**, даже если в списке одно значение: у произведения может быть несколько авторов, издания на нескольких языках и несколько издателей. Это придётся учесть, когда записи будут превращаться в таблицу (часть 6). Некоторых полей у записи может не быть вовсе, поэтому для них используется метод `.get()`.

Как работают условия в параметре `q`, проще всего понять, сравнив число найденных записей. Ниже то же слово дополнено условием на язык.

In [ ]:
# Условие на поле добавляется к слову через пробел; limit=0 — нужны не записи, а только их число
params = {"q": "Chekhov language:ger", "limit": 0}

get_json(OPENLIBRARY_API, params, headers=OL_HEADERS)["numFound"]

#### ✏️📨🚫 Задание 6

1. Отправьте запрос к Open Library по теме из вашей области и выведите общее число записей, а также названия и первых авторов первых десяти записей. Попробуйте записать запрос и латиницей, и кириллицей: описания в каталоге в основном составлены на английском языке, но названия книг на русском тоже встречаются.
2. Уточните запрос с помощью одного из приёмов, перечисленных в таблице и в списке выше: условия на поле, диапазона лет, точной фразы или параметра `sort`. Как изменились число записей и состав первых десяти?

📨 Отправьте в телеграм-чат курса оба запроса, число записей по каждому из них и одну запись, которая показалась вам неожиданной.

In [ ]:
# ВАШ КОД

#### ✏️🔁🚫 Задание 7

Одно из правил вежливого сбора требует не скачивать одно и то же дважды. Сохраните ответ из предыдущего задания (словарь `data`) в файл `openlibrary_response.json` функцией `json.dump()`, затем прочитайте файл обратно функцией `json.load()` и выведите из прочитанного словаря `numFound`.

*Подсказка:* чтобы кириллица и другие алфавиты сохранились в читаемом виде, передайте в `json.dump()` аргумент `ensure_ascii=False`; аргумент `indent=2` расставляет отступы, и файл становится удобным для чтения.

In [ ]:
# ВАШ КОД

---
## Часть 5. Все записи из Open Library: пагинация смещением

### Три способа пагинации

На занятии встречаются три способа указать, какая порция нужна.

| Способ | Как задаётся порция | Где встречается | Особенности |
|---|---|---|---|
| курсор | метка, которую присылает сервер | MediaWiki (`continue`) | порции не пропадают и не повторяются, даже если данные меняются во время сбора; нельзя сразу перейти к сотой порции |
| смещение | сколько записей пропустить от начала | Open Library (`offset`) | можно запросить любую порцию; если во время сбора в выдаче появятся новые записи, часть записей сдвинется и может повториться или пропасть |
| номер страницы | порядковый номер порции | Open Library (`page`), GitHub | то же смещение, только выраженное в порциях: `page=3` при `limit=100` — то же, что `offset=200` |

Какой способ доступен, определяет владелец API, и узнать это можно только из документации. В Open Library параметр `limit` задаёт размер порции, а параметр `offset` — сколько записей пропустить от начала. Смещение отсчитывается от нуля:

| Порция | `offset` | Записи |
|---|---|---|
| первая | 0 | 1–100 |
| вторая | 100 | 101–200 |
| третья | 200 | 201–300 |

В других API смещение может отсчитываться от единицы, поэтому начало отсчёта всегда проверяется по документации или пробным запросом.

### Цикл со смещением

Цикл устроен так же, как в части 2, но два шага отличаются:

- чтобы получить следующую порцию, к `offset` прибавляется размер порции, а не вызывается `params.update(...)`;
- условием выхода служит сравнение числа собранных записей с `numFound`, а не проверка ключа `continue`.

Прежде чем писать цикл, полезно узнать, что произойдёт, если смещение окажется больше числа найденных записей.

In [ ]:
params = {"q": "Chekhov", "limit": 10, "offset": 100000, "fields": FIELDS}

# Здесь нарочно requests.get, а не get_json: нужен сам ответ со статус-кодом
response = requests.get(OPENLIBRARY_API, params=params, headers=OL_HEADERS, timeout=10)

response.status_code

In [ ]:
# Ошибки нет, но список записей пуст
len(response.json()["docs"])

Сервер не сообщает об ошибке: он отвечает кодом `200` и присылает пустой список. Поэтому в цикле нужна проверка на пустую порцию; без неё ошибка в расчёте смещения превратилась бы в серию бесполезных запросов.

### Предел у смещения

На каждый запрос со смещением сервер заново находит и пропускает все предыдущие записи, и на больших смещениях это обходится дорого. Поэтому многие API ограничивают смещение. В документации Open Library такого предела нет, но действует правило из части 4: большие объёмы данных скачивают из выгрузок, а не через API.

Размер порции тоже влияет на нагрузку. Чем больше `limit`, тем меньше запросов, а если в `fields` перечислено немного полей, каждый ответ остаётся небольшим. Поэтому в функции ниже по умолчанию запрашивается порция в 500 записей.

#### ✏️💬 Задание 8

Допишите функцию `search_openlibrary()`. В цикле она должна выполнять четыре шага:

1. отправить запрос с текущим `offset` через `get_json()` с заголовком `OL_HEADERS`;
2. добавить записи из `docs` в общий список;
3. выйти из цикла, если собрано не меньше `min(numFound, max_records)` записей или если пришла пустая порция;
4. в противном случае прибавить `limit` к `offset` и сделать паузу.

Сначала проверьте функцию на небольшом числе записей: `limit=100, max_records=250` (три порции). Затем в следующей ячейке соберите все записи по запросу `Chekhov` и сравните их число с `numFound`.

In [ ]:
def search_openlibrary(query: str, max_records: int = 10000, limit: int = 500, max_requests: int = 20) -> list[dict]:
    """Возвращает записи Open Library по запросу (не больше max_records)."""
    records = []
    offset = 0

    for _ in range(max_requests):
        params = {"q": query, "limit": limit, "offset": offset, "fields": FIELDS}
        # ВАШ КОД
        ...

    return records


# Проверка на трёх порциях
test = search_openlibrary("Chekhov", max_records=250, limit=100)
len(test)

In [ ]:
# Все записи по запросу: около десятка запросов по 500 записей
chekhov = search_openlibrary("Chekhov")
len(chekhov)

In [ ]:
# Если функция из задания 8 пока не работает, раскомментируйте строки ниже,
# чтобы продолжить занятие: они скачают первые 500 записей одной порцией
# backup = get_json(OPENLIBRARY_API, {"q": "Chekhov", "limit": 500, "fields": FIELDS}, headers=OL_HEADERS)
# chekhov = backup["docs"]

---
## Часть 6. Таблица

### Функция isinstance()

In [ ]:
def describe(value):
    """Сообщает, что за значение передано."""
    if isinstance(value, list):
        return f"список из {len(value)} элементов"
    if value is None:
        return "пустое значение"
    return f"значение {value}"


print(describe(["rus", "eng"]))
print(describe(None))
print(describe(1904))

### Как превратить список в одно значение

Для анализа нужна таблица, в каждой ячейке которой лежит одно значение, а часть полей записей Open Library представляет собой списки. Самое простое решение состоит в том, чтобы брать первый элемент списка, а если поля нет, записывать `None`.

#### ✏️🚫 Задание 9

Допишите функцию `first(value)`: если `value` — непустой список, она возвращает его первый элемент, а в остальных случаях возвращает `None`. Проверьте её на трёх значениях: `["eng"]`, `[]` и `None`.

In [ ]:
def first(value):
    """Первый элемент списка или None."""
    # ВАШ КОД
    ...


print(first(["eng"]), first([]), first(None))

Для языков одного первого элемента недостаточно: у произведения бывают издания на нескольких языках, а порядок языков в списке ничего не означает. Поэтому в таблицу попадут и первый язык, и число языков. Функция `count_items()` возвращает длину списка, а если поля нет, возвращает ноль.

Это решение не единственное. Можно было бы склеить все языки в одну строку через запятую или завести отдельный столбец для каждого языка. Выбор зависит от вопроса: чтобы посчитать, у скольких произведений есть издания на немецком языке, понадобился бы полный список, а для первого обзора достаточно первого языка и числа языков.

In [ ]:
def count_items(value) -> int:
    """Длина списка или 0, если значение не список."""
    if isinstance(value, list):
        return len(value)
    return 0


print(count_items(["rus", "eng"]), count_items(None))

### Имя и идентификатор автора

У каждой записи есть два поля, относящихся к автору. Поле `author_name` содержит имя в том виде, в каком оно записано в каталоге, а поле `author_key` — идентификатор записи об авторе. Имена пишутся по-разному: «Антон Павлович Чехов», «Anton Chekov», «A. P. Chekhov». Идентификатор у одной записи об авторе всегда один и тот же; у основной записи об Антоне Чехове это `OL19677A`. По идентификатору можно открыть страницу автора в каталоге.

In [ ]:
record = chekhov[0]

# Списки имён и идентификаторов идут в одном порядке
print(record.get("author_name"))
print(record.get("author_key"))

In [ ]:
# Адрес страницы автора в каталоге составляется из идентификатора
print("https://openlibrary.org/authors/" + first(record.get("author_key")))

Идентификаторы вместо имён лежат в основе связанных данных, о которых пойдёт речь на занятии 5 (Wikidata). Пока достаточно добавить в таблицу столбец, который показывает, указан ли среди авторов записи Антон Чехов. Оператор `in` проверяет, есть ли значение в списке.

### ⭐ Авторитетные записи

Запись об авторе, на которую ссылается `author_key`, в библиотечном деле называется **авторитетной записью** (authority record). В ней закреплена одна принятая форма имени и перечислены варианты, по которым читатель может искать того же человека. Национальные библиотеки ведут собственные авторитетные файлы; например, Библиотека Конгресса США ведёт LC Name Authority File. Сводный файл VIAF (Virtual International Authority File) связывает записи разных национальных библиотек об одном человеке, а Wikidata всё чаще связывает библиотечные базы с небиблиотечными.

Запись об авторе в Open Library хранит ссылки на эти файлы в поле `remote_ids`. Полная запись загружается отдельным запросом по адресу `https://openlibrary.org/authors/<идентификатор>.json`.

In [ ]:
# ⭐ Полная запись об авторе: варианты имени и ссылки на внешние авторитетные файлы
author = get_json("https://openlibrary.org/authors/OL19677A.json", {}, headers=OL_HEADERS)

author.keys()

In [ ]:
# ⭐ Идентификаторы того же человека во внешних базах: Wikidata, VIAF, Библиотека Конгресса и другие
author.get("remote_ids")

In [ ]:
# ⭐ Сколько вариантов имени хранит запись и какие из них идут первыми
print(len(author.get("alternate_names", [])))
print(author.get("alternate_names", [])[:10])

Среди вариантов имени стоит поискать ошибки. Записи об авторах пополняются и автоматически, и вручную, поэтому в список иногда попадают имена других людей, а вместе с ними Чехову могут оказаться приписаны чужие произведения.

### Из записей — таблица

Сначала преобразование проверяется на одной записи: из неё составляется словарь, который станет строкой будущей таблицы.

In [ ]:
# Словарь с одним значением в каждом поле
{
    "key": record["key"],
    "title": record.get("title"),
    "author": first(record.get("author_name")),
    # get с пустым списком: если поля нет, in вернёт False, а не вызовет ошибку
    "by_chekhov": "OL19677A" in record.get("author_key", []),
    "year": record.get("first_publish_year"),
    "language": first(record.get("language")),
    "n_languages": count_items(record.get("language")),
    "publisher": first(record.get("publisher")),
    "editions": record.get("edition_count"),
    "link": "https://openlibrary.org" + record["key"],
}

Затем то же преобразование выполняется в цикле для всех записей.

In [ ]:
rows = []

for record in chekhov:
    rows.append({
        "key": record["key"],
        "title": record.get("title"),
        "author": first(record.get("author_name")),
        "by_chekhov": "OL19677A" in record.get("author_key", []),
        "year": record.get("first_publish_year"),
        "language": first(record.get("language")),
        "n_languages": count_items(record.get("language")),
        "publisher": first(record.get("publisher")),
        "editions": record.get("edition_count"),
        "link": "https://openlibrary.org" + record["key"],
    })

df = pd.DataFrame(rows)

df.head()

In [ ]:
# Год пришёл числом, но из-за пропусков pandas хранит его как дробное число (float64)
print(df["year"].dtype)

# Тип "Int64" с заглавной буквы — целые числа, среди которых допускаются пропуски
df["year"] = df["year"].astype("Int64")
print(df["year"].dtype)

### 🔁 pandas: что понадобится дальше

Эти методы встречались в курсе «Цифровые технологии в профессиональной деятельности» и на занятии 1.

| Метод | Что делает |
|---|---|
| `df["столбец"].value_counts()` | считает, сколько раз встречается каждое значение |
| `df.sample(20, random_state=1)` | выбирает 20 случайных строк; `random_state` делает выборку воспроизводимой |
| `df["столбец"].notna()` | возвращает `True` там, где значение есть |
| `.mean()` от столбца из `True` и `False` | считает долю значений `True` |
| `df.to_csv("файл.csv", index=False)` | сохраняет таблицу в файл |

In [ ]:
df["language"].value_counts().head(10)

In [ ]:
df.to_csv("chekhov_openlibrary.csv", index=False, encoding="utf-8")

---
## Часть 7. Что в таблице говорит о Чехове, а что — о каталоге

Таблица собрана. Прежде чем отвечать по ней на вопрос, как Чехов представлен в библиотечном каталоге, нужно задать три вопроса о самом наборе данных.

1. **Точность: всё ли здесь относится к Чехову?** В выдачу попадают записи, в которых встречается слово, а не записи о человеке, поэтому совпадения бывают случайными.
2. **Полнота: всё ли о Чехове здесь есть?** Имя пишется по-разному на разных языках и иногда с ошибками, поэтому по одному написанию находится не всё.
3. **Источник: почему записи распределены именно так?** Число книг на том или ином языке зависит не только от интереса к Чехову, но и от того, какие библиотеки и книготорговцы передали описания в каталог и насколько подробно эти описания составлены.

Эти же три вопроса вам предстоит задать собственному набору данных в домашнем задании 1.

### Точность

#### ✏️📨🚫 Задание 10

Выведите 20 случайных записей со столбцами `title`, `author`, `by_chekhov`, `year` и `link`. Для каждой записи решите, относится ли она к Антону Чехову, то есть написана ли книга им или о нём, и запишите решения списком `relevant` из нулей и единиц в том же порядке. Если по названию решить нельзя, откройте ссылку из столбца `link`. Посчитайте долю релевантных записей: это оценка **точности** запроса. Какие ложные совпадения встретились?

📨 Отправьте в телеграм-чат курса долю релевантных записей и два-три примера ложных совпадений.

In [ ]:
# random_state фиксирует случайную выборку: при повторном запуске выпадут те же записи
sample = df.sample(20, random_state=1)[["title", "author", "by_chekhov", "year", "link"]]
sample

In [ ]:
relevant = [
    # ВАШ КОД: 20 значений 0 или 1
]

# ВАШ КОД: доля релевантных записей

Столбец `by_chekhov` решает задачу лишь отчасти: он отмечает книги, автором которых указан Чехов. Книги о Чехове отмечены в нём значением `False`, хотя к вопросу они относятся. Посторонние записи бывают разными: книги однофамильцев (например, актёра Михаила Чехова, племянника писателя), антологии, в которых рассказ Чехова — один из многих, учебники, в описании которых упомянуто его имя.

In [ ]:
# Доля записей, среди авторов которых есть основная запись об Антоне Чехове
df["by_chekhov"].mean()

### Полнота

Чтобы сравнить разные написания, не нужно скачивать сами записи: достаточно узнать `numFound`. Если указать `limit=0`, Open Library присылает только число найденных записей. В документации этот приём прямо не описан, но пробный запрос показывает, что он работает. Такие наблюдения стоит перепроверять, потому что недокументированное поведение API может измениться без предупреждения.

In [ ]:
def count_results(query: str) -> int:
    """Число записей Open Library по запросу."""
    # limit=0: сами записи не нужны, нужно только их число
    params = {"q": query, "limit": 0}
    data = get_json(OPENLIBRARY_API, params, headers=OL_HEADERS)

    return data["numFound"]


count_results("Chekhov")

#### ✏️📨💬 Задание 11

Сравните число записей для разных написаний имени: `Chekhov`, `Chekov` (частая ошибка), `Tchekhov` (французское), `Tschechow` (немецкое), `Čechov` (чешское и итальянское), `Csehov` (венгерское), `Czechow` (польское) и `Чехов`. Выведите результаты таблицей и ответьте на вопросы.

- В записи об авторе каталог хранит десятки вариантов имени, поэтому многие написания дают похожие числа. Но находятся ли по ним те же записи? Соберите функцией `search_openlibrary()` записи по одному из написаний и посчитайте, сколько из них нет в `df`. *Подсказка:* выражение `known = set(df["key"])` создаёт множество уже собранных идентификаторов, и проверка `key not in known` выполняется быстро.
- При каком написании в выдачу попадает много посторонних записей? *Подсказка:* посмотрите на названия; по-польски «Czechów» означает ещё и «чехов», то есть жителей Чехии.
- Запросы можно объединять оператором `OR`, записанным заглавными буквами: `Chekhov OR Tschechow`. Составьте запрос, по которому находится как можно больше записей о Чехове и как можно меньше лишних. Сколько записей по нему находится и сколько запросов по 500 записей понадобится, чтобы собрать их все?

📨 Отправьте в телеграм-чат курса свой составной запрос, число записей по нему и объяснение, почему одни написания вы включили, а другие нет.

In [ ]:
variants = ["Chekhov", "Chekov", "Tchekhov", "Tschechow", "Čechov", "Csehov", "Czechow", "Чехов"]

# ВАШ КОД

### Источник

Теперь нужно выяснить, откуда пришли описания. Сначала полезно посмотреть, как записи распределены по языкам и издателям.

In [ ]:
df["language"].value_counts().head(10)

In [ ]:
# Записи без указания языка
df["language"].isna().sum()

In [ ]:
df["publisher"].value_counts().head(10)

Источник описания виден на уровне изданий. Список изданий любого произведения можно получить в формате JSON: для этого к адресу страницы произведения добавляется окончание `/editions.json`. Ниже загружаются издания первого произведения из таблицы.

In [ ]:
editions_url = "https://openlibrary.org" + df.loc[0, "key"] + "/editions.json"

editions = get_json(editions_url, {"limit": 50}, headers=OL_HEADERS)

editions.keys()

In [ ]:
# Издания лежат в entries
len(editions["entries"])

In [ ]:
# У каждого издания есть список источников описания
for edition in editions["entries"]:
    print(edition.get("source_records"))

Источник записан в виде `префикс:подробности`. Префикс `marc` означает запись из библиотечного каталога (MARC — стандартный формат библиотечных описаний), префикс `ia` — книгу, оцифрованную Internet Archive, префиксы `amazon` и `bwb` — данные книготорговцев Amazon и Better World Books, а префикс `promise` — партии книг, переданные Better World Books. У одного издания бывает несколько источников. Ниже префиксы подсчитываются по шагам.

In [ ]:
prefixes = []

for edition in editions["entries"]:
    # Если у издания нет source_records, цикл по пустому списку ничего не добавит
    for source in edition.get("source_records", []):
        # split(":") делит строку по двоеточию; [0] — часть до первого двоеточия
        prefixes.append(source.split(":")[0])

pd.Series(prefixes).value_counts()

#### ✏️📨🚫 Задание 12

1. Посчитайте долю записей, у которых указан год, долю записей, у которых указан автор, и долю записей, у которых указан язык.
2. Ответьте в ячейке ниже в 4–6 предложениях: что по распределению записей между языками и издателями можно узнать о Чехове, а что — об Open Library? Можно ли по столбцу `year` построить «хронологию интереса к Чехову»? Учтите, что `first_publish_year` — год первого известного каталогу издания произведения, а не год каждого издания.

📨 Отправьте ответ в телеграм-чат курса: на следующем занятии несколько ответов будут разобраны вместе.

In [ ]:
# ВАШ КОД

*Ваш ответ:*

---
## Часть 8. ⭐ Практикум продвинутого уровня

В практикуме отрабатывается исследовательская работа с собранными данными: как поставить вопрос, на который можно ответить по этим данным, как выдвинуть гипотезу до расчёта, как её проверить и как сформулировать ответ с учётом того, как устроен источник.

Задачи расположены по нарастающей сложности. В задачах П1–П3 используется уже собранная таблица `df` и список записей `chekhov`, новых запросов не нужно. В задаче П4 данные собираются заново, в задачах П5 и П6 соединяются два источника, Википедия и Open Library, а в задаче П7 исследовательский вопрос вы формулируете сами. Решайте задачи по порядку, сколько успеете; задачи П5 и П6 связаны между собой.

Работа над каждой задачей строится одинаково:

1. **Гипотеза.** Прежде чем считать, запишите, какой ответ вы ожидаете и почему. Гипотеза, записанная после расчёта, ничего не проверяет.
2. **Расчёт.** Выполните шаги, перечисленные в задаче.
3. **Ответ.** Ответьте на вопрос в 2–4 предложениях. Отделите то, что результат говорит о предмете (о Чехове и его рецепции), от того, что он говорит об источнике (о каталоге или Википедии). Если гипотеза не подтвердилась, объясните, почему.

Гипотеза и ответ записываются в markdown-ячейке под расчётом. В чат отправляются вопрос, гипотеза, главный результат и ответ.

Для задачи П4 понадобится функция, которая собирает записи по любому набору параметров, а не только по строке запроса. Она обобщает функцию `search_openlibrary()` из задания 8.

In [ ]:
def search_works(params: dict, max_records: int = 10000, limit: int = 500, max_requests: int = 30) -> list[dict]:
    """⭐ Собирает записи Open Library по произвольным параметрам поиска."""
    records = []
    offset = 0

    for _ in range(max_requests):
        # Копия словаря: исходные параметры вызывающего кода не меняются
        page_params = dict(params)
        page_params.update({"limit": limit, "offset": offset, "fields": FIELDS})

        data = get_json(OPENLIBRARY_API, page_params, headers=OL_HEADERS)
        docs = data.get("docs", [])
        records.extend(docs)

        if len(records) >= min(data["numFound"], max_records) or not docs:
            break

        offset += limit
        time.sleep(1)

    return records[:max_records]


# Параметр author_key отбирает произведения одной авторитетной записи
len(search_works({"author_key": "OL19677A"}, max_records=600))

### ⭐ Чтение

К этой части прилагаются два текста для ознакомления по желанию.

- Drucker J. Humanities Approaches to Graphical Display // Digital Humanities Quarterly. 2011. Vol. 5, № 1. Автор предлагает говорить о данных как о capta, то есть о «взятом», а не о «данном»: любой набор данных возникает в результате решений того, кто его собирал.
- The Santa Barbara Statement on Collections as Data (2017). Это короткий манифест проекта Collections as Data о том, как библиотеки и архивы могут предоставлять свои собрания исследователям в виде данных и что при этом теряется.

Тем, кого тема заинтересует, можно рекомендовать книгу: Thylstrup N. B. The Politics of Mass Digitization. Cambridge (MA): MIT Press, 2018.

#### ✏️⭐📨💬 Задание П1. Что каталог считает первым изданием

**Вопрос.** Какая доля произведений Чехова, по данным Open Library, была впервые издана при его жизни, то есть не позже 1904 года?

**Шаги.**

1. Запишите гипотезу: какую долю вы ожидаете?
2. Отберите из таблицы `df` произведения Чехова, у которых указан год. Условия объединяются оператором `&`.
3. Посчитайте долю произведений, у которых `year` не больше 1904. Сравнение `works["year"] <= 1904` даёт столбец из `True` и `False`, а метод `.mean()` превращает его в долю.
4. Выведите десять произведений с наибольшим числом изданий (`editions`) вместе с годом. Сверьте год у нескольких известных пьес с годом их первой публикации по Википедии.

**Ответ.** Что на самом деле измеряет поле `first_publish_year`? Почему результат так сильно расходится с биографией Чехова, хотя все его произведения были написаны до 1904 года?

In [ ]:
works = df[df["by_chekhov"] & df["year"].notna()]

# ВАШ КОД: доля произведений с годом не позже 1904

In [ ]:
# ВАШ КОД: десять произведений с наибольшим числом изданий

*Гипотеза:*

*Ответ:*

#### ✏️⭐📨💬 Задание П2. Канон по числу изданий

**Вопрос.** Какие произведения Чехова каталог считает самыми издаваемыми и совпадает ли этот список с тем, что принято считать чеховским каноном?

**Шаги.**

1. Запишите гипотезу: назовите пять произведений, которые, по-вашему, окажутся наверху списка.
2. Выведите 15 произведений Чехова с наибольшим числом изданий (столбцы `title`, `year`, `editions`).
3. Для каждой строки решите, что она описывает: отдельное произведение (`work`), сборник произведений Чехова (`collection`) или антологию, где Чехов — один из многих авторов (`anthology`). Запишите решения списком `kinds` в том же порядке и добавьте его в таблицу новым столбцом.
4. Посчитайте, сколько строк каждого вида попало в первые 15, методом `.value_counts()`.

**Ответ.** Совпал ли список с вашей гипотезой? Что в нём говорит о популярности произведений, а что — о том, как каталог понимает слово «произведение»?

In [ ]:
# ВАШ КОД: 15 произведений Чехова с наибольшим числом изданий

In [ ]:
kinds = [
    # ВАШ КОД: 15 значений "work", "collection" или "anthology"
]

# ВАШ КОД: новый столбец и подсчёт видов

*Гипотеза:*

*Ответ:*

#### ✏️⭐📨💬 Задание П3. На каких языках Чехова читают и на каких о нём пишут

**Вопрос.** Различается ли языковой состав книг Чехова и книг о Чехове?

**Шаги.**

1. Запишите гипотезу: какие языки, по-вашему, будут первыми в каждой из двух групп?
2. Для этого задания нужен полный список языков каждой записи, а не первый язык из таблицы, поэтому работать удобнее со списком записей `chekhov`. Заведите два счётчика `Counter` из модуля `collections` (стандартная библиотека): он считает, сколько раз встретилось каждое значение.

   ```python
   from collections import Counter

   by_counter = Counter()
   by_counter.update(["eng", "rus"])   # update добавляет к счётчику все элементы списка
   by_counter.most_common(3)           # самые частые значения
   ```

3. В цикле по записям `chekhov` определите группу записи (есть ли `"OL19677A"` в `author_key`) и добавьте её языки в нужный счётчик.
4. Сведите в одну таблицу по восемь самых частых языков каждой группы: язык, число произведений и доля от числа записей группы.

**Ответ.** Подтвердилась ли гипотеза? Учтите, что язык считается один раз на произведение, сколько бы изданий на этом языке у него ни было. Что этот способ подсчёта делает с русским языком, на котором у каждого произведения Чехова обычно много изданий?

In [ ]:
from collections import Counter

by_counter = Counter()
about_counter = Counter()

# ВАШ КОД: цикл по записям chekhov и итоговая таблица

*Гипотеза:*

*Ответ:*

#### ✏️⭐📨💬 Задание П4. Чехов на фоне современников

**Вопрос.** Какие черты «каталожного профиля» Чехова свойственны только ему, а какие общие для русских писателей XIX века и для каталога в целом?

Здесь другие писатели служат **базовой линией**, то есть точкой отсчёта: если черта есть у всех, она скорее объясняется устройством каталога.

**Шаги.**

1. Запишите гипотезу: чем, по-вашему, профиль Чехова будет отличаться от профилей Тургенева, Толстого и Достоевского?
2. Ниже дан словарь `authors` с идентификаторами авторитетных записей и годами смерти. Идентификаторы найдены поиском по авторам (`https://openlibrary.org/search/authors.json?q=Turgenev`); при желании добавьте одного нерусского писателя того же времени.
3. Напишите функцию `author_profile(author_key, death_year)`, которая собирает произведения функцией `search_works({"author_key": author_key})` и возвращает словарь:
   - `works` — число произведений;
   - `share_eng` и `share_rus` — доли произведений, у которых среди языков есть английский и русский;
   - `median_year` — медиана года первого издания (вычисляется через `pd.Series(years).median()`);
   - `share_lifetime` — доля произведений с годом первого издания не позже года смерти автора (среди тех, у кого год указан).
4. Сведите профили в таблицу и округлите доли методом `.round(2)`.

**Ответ.** Какие черты общие для всех авторов, а какие отличают Чехова? Что из этого вы отнесёте к каталогу, а что — к рецепции?

In [ ]:
authors = {
    "Чехов": ("OL19677A", 1904),
    "Тургенев": ("OL10240A", 1883),
    "Толстой": ("OL26783A", 1910),
    "Достоевский": ("OL22242A", 1881),
}


def author_profile(author_key: str, death_year: int) -> dict:
    """Каталожный профиль автора по его авторитетной записи."""
    works = search_works({"author_key": author_key})

    # ВАШ КОД

    return {}


# ВАШ КОД: таблица профилей

*Гипотеза:*

*Ответ:*

#### ✏️⭐📨💬 Задание П5. Пьесы Чехова в Википедии

**Вопрос.** Какие пьесы Чехова наиболее заметны в Википедии и одинаково ли они заметны в русском разделе и в других языковых разделах?

Заметность можно измерить двумя способами: длиной статьи в русской Википедии (в байтах) и числом языковых разделов, где о пьесе есть статья.

**Шаги.**

1. Запишите гипотезу: какие пьесы окажутся наверху по каждому из двух показателей и совпадут ли эти списки?
2. Соберите статьи «Категории:Пьесы Антона Чехова» функцией `get_all_category_members()`. Статью о сборнике («Пьесы (сборник)») исключите.
3. Для каждой статьи отправьте запрос с такими параметрами:

   ```python
   params = {
       "action": "query",
       "prop": "langlinks|info",   # ссылки на другие языковые разделы и сведения о странице
       "lllimit": "max",           # все языковые ссылки, а не первые 10
       "titles": title,
       "format": "json",
       "formatversion": 2,         # pages — список, у языковой ссылки ключи lang и title
   }
   ```

   Страница лежит в `data["query"]["pages"][0]`. Из неё нужны длина статьи (`page["length"]`), число языковых ссылок (`len(page.get("langlinks", []))`) и английское название пьесы — значение `title` у той ссылки, у которой `lang` равен `"en"` (если такой ссылки нет, запишите `None`).
4. Соберите таблицу `wiki_df` со столбцами `ru_title`, `en_title`, `length`, `langlinks`. Добавьте места в двух рядах: `wiki_df["length"].rank(ascending=False)` и то же для `langlinks`.

**Ответ.** Совпадают ли два ряда? Какие пьесы заметнее в русском разделе, чем в других, и наоборот? С чем это может быть связано?

Таблица `wiki_df` понадобится в задаче П6.

In [ ]:
plays = get_all_category_members("Категория:Пьесы Антона Чехова")

wiki_rows = []

# ВАШ КОД: цикл по статьям, запрос с prop=langlinks|info, словарь для каждой пьесы

wiki_df = pd.DataFrame(wiki_rows)

*Гипотеза:*

*Ответ:*

#### ✏️⭐📨💬 Задание П6. Википедия и каталог: объединение двух источников

**Вопрос.** Совпадает ли канон Википедии с каноном издателей, то есть часто ли переиздаются те пьесы, о которых пишут во многих языковых разделах?

Чтобы ответить, нужно соединить две таблицы: `wiki_df` и таблицу с данными Open Library. Общего идентификатора у них нет, поэтому связующим звеном служит английское название пьесы.

**Шаги.**

1. Запишите гипотезу: будет ли связь между числом языковых разделов и числом изданий сильной, слабой или её не будет?
2. Английские названия в Википедии часто содержат уточнение в скобках: «Ivanov (play)». Уберите его регулярным выражением (повторение из курса 1):

   ```python
   import re
   re.sub(r"\s*\(.*?\)", "", "Ivanov (play)")   # 'Ivanov'
   ```

3. Для каждой пьесы с английским названием отправьте запрос к Open Library с параметрами `title` (название без скобок), `author_key` (`"OL19677A"`), `limit` (100) и `fields` (`"key,title,edition_count"`). Из ответа возьмите число найденных произведений (`numFound`) и сумму изданий (`edition_count`) по всем найденным произведениям.
4. Соберите таблицу `ol_df` со столбцами `en_title` (исходное название со скобками, чтобы оно совпадало с `wiki_df`), `ol_works` и `ol_editions`.
5. Соедините таблицы функцией `pd.merge()`:

   ```python
   merged = pd.merge(wiki_df, ol_df, on="en_title", how="left")
   ```

   Аргумент `on` указывает столбец, по которому соединяются строки, а `how="left"` сохраняет все строки левой таблицы (`wiki_df`), даже если в правой для них ничего не нашлось.
6. Посчитайте коэффициент ранговой корреляции Спирмена между `langlinks` и `ol_editions`. Он равен обычной корреляции между местами в двух рядах:

   ```python
   merged["langlinks"].rank().corr(merged["ol_editions"].rank())
   ```

   Значения близки к 1, если ряды совпадают, к 0, если связи нет, и к −1, если ряды противоположны.
7. Найдите пьесы, для которых в каталоге ничего не нашлось или нашлось подозрительно мало. Для двух-трёх из них попробуйте другие варианты английского названия (например, «Swan Song» вместо «Swansong») и посмотрите, как изменится результат.

**Ответ.** В 4–5 предложениях ответьте на вопрос задачи. Какие расхождения объясняются популярностью пьес, а какие — способом соединения таблиц (разные переводы названий, сборники «and other plays», дробление одной пьесы на много записей)?

In [ ]:
import re

ol_rows = []

# ВАШ КОД: запрос к Open Library для каждого английского названия

ol_df = pd.DataFrame(ol_rows)

# ВАШ КОД: pd.merge, корреляция Спирмена, разбор пьес без пары

*Гипотеза:*

*Ответ:*

#### ✏️⭐📨💬 Задание П7. Собственный исследовательский вопрос

**Задача.** Сформулируйте собственный исследовательский вопрос, на который можно ответить по таблицам этого занятия (`df`, `wiki_df`, `merged`) или по данным, которые можно собрать теми же функциями. Ответьте на него тем же порядком: гипотеза, расчёт, ответ с учётом устройства источников.

**Требования к вопросу.**

- Вопрос касается гуманитарного предмета (писателя, произведений, рецепции, переводов), а не только устройства API.
- Ответить на него можно по данным, которые у вас есть или которые можно собрать за 10–20 запросов.
- В ответе есть число или таблица.

**Примеры вопросов** (их можно взять за образец, но лучше придумать свой):

- Чаще ли издаются в переводах большие пьесы Чехова, чем одноактные? (Понадобится разметить пьесы в `merged` вручную.)
- Связана ли длина статьи в русской Википедии с числом языков, на которых пьеса издана в Open Library?
- Похож ли «канон» рассказов Чехова в Википедии (Категория:Рассказы Антона Чехова, 186 статей) на канон пьес?

**Ответ.** Вопрос, гипотеза, код, главный результат и ответ в 4–6 предложениях с ограничениями.

📨 Отправьте в чат свой вопрос и ответ: на следующем занятии несколько вопросов будут разобраны вместе.

In [ ]:
# ВАШ КОД

*Гипотеза:*

*Ответ:*

---
## Итоги

На занятии пройден полный путь сбора данных из API:

`вопрос` → `документация и правила доступа` → `все порции` → `повтор при сбоях` → `таблица` → `проверка точности` → `проверка полноты` → `интерпретация с учётом источника` → `CSV`

Технически новыми были пагинация двумя способами (курсором и смещением), обработка ошибок и правила доступа к API: заголовок User-Agent с контактом, лимиты запросов, ключи и их хранение. Кроме того, на занятии было видно, что параметры запроса не берутся готовыми: их выбирают по документации из нескольких вариантов, и этот выбор влияет на то, что окажется в наборе данных.

Вторая половина пути важна не меньше первой. Таблица, полученная из API, отвечает на конкретный запрос к конкретному собранию данных. На примере Чехова видно, что от написания имени меняется состав выдачи, а распределение записей по языкам больше связано с тем, откуда каталог получает описания, чем с рецепцией писателя.

### Документация

- MediaWiki API, продолжение запросов: https://www.mediawiki.org/wiki/API:Continue
- MediaWiki API, список элементов категории: https://www.mediawiki.org/wiki/API:Categorymembers
- Лимиты Wikimedia APIs: https://www.mediawiki.org/wiki/Wikimedia_APIs/Rate_limits
- Open Library, правила для разработчиков: https://openlibrary.org/developers/api
- Open Library Search API: https://openlibrary.org/dev/docs/api/search
- requests, ошибки и исключения: https://requests.readthedocs.io/en/latest/user/quickstart/#errors-and-exceptions
- Python, обработка исключений: https://docs.python.org/3/tutorial/errors.html